# CHAPTER 4. 국가법령정보 Open API — Legal Constraint 분석

## 분석 목적

앞선 상담 데이터가
**"업무 수행에 어떤 데이터가 필요한가"**를 보여줬다면,

이번 법령 데이터는
**"그 데이터를 실제로 처리할 때 법적으로 무엇을 지켜야 하는가"**를 정의한다.

따라서 본 분석의 목적은 법령을 단순 요약하는 것이 아니라,

법령 조문
→ 개인정보·신용정보 처리 의무 추출
→ Constraint 구조화
→ Gateway의 ALLOW / TRANSFORM / BLOCK / REVIEW 조건

으로 변환하는 것이다.

## 핵심 질문

1. 개인정보·신용정보는 어떤 목적에서 이용할 수 있는가?
2. 목적에 필요한 최소 범위는 어디까지인가?
3. 제3자 또는 외부 SaaS·AI로 전달할 때 어떤 조건이 필요한가?
4. 가명처리가 요구되거나 허용되는 조건은 무엇인가?
5. 가명정보의 재식별 위험은 어떻게 통제해야 하는가?
6. 외부 전송 시 어떤 안전조치와 기록이 필요한가?

## 최종 목표

Legal Constraint를 다음 구조로 만든다.

Law
→ Article
→ Obligation
→ Condition
→ Data Type
→ Required Treatment
→ Gateway Decision

In [3]:
# 01. 라이브러리 불러오기

import requests
import pandas as pd
import xml.etree.ElementTree as ET

OC = "kimnahee_fpg"

SEARCH_URL = "https://www.law.go.kr/DRF/lawSearch.do"
SERVICE_URL = "https://www.law.go.kr/DRF/lawService.do"

In [4]:
# 02. 법령 검색 함수

def search_law(law_name):

    params = {
        "OC": OC,
        "target": "eflaw",
        "type": "XML",
        "search": 1,
        "query": law_name,
        "nw": 3,
        "display": 100
    }

    response = requests.get(
        SEARCH_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    root = ET.fromstring(response.content)

    rows = []

    for law in root.findall(".//law"):

        row = {}

        for child in law:
            row[child.tag] = "".join(child.itertext()).strip()

        rows.append(row)

    return pd.DataFrame(rows)

In [5]:
privacy_search = search_law("개인정보 보호법")

credit_search = search_law(
    "신용정보의 이용 및 보호에 관한 법률"
)

display(privacy_search)
display(credit_search)

,법령일련번호,현행연혁코드,법령명한글,법령약칭명,법령ID,공포일자,공포번호,제개정구분명,소관부처코드,소관부처명,법령구분명,공동부령정보,시행일자,자법타법여부,법령상세링크
0,270351,현행,개인정보 보호법,,011357,20250401,20897,일부개정,1790365,개인정보보호위원회,법률,,20251002,,/DRF/lawService.do?OC=kimnahee_fpg&target=efla...
1,283503,현행,개인정보 보호법 시행령,,011468,20260219,36121,일부개정,1790365,개인정보보호위원회,대통령령,,20260820,,/DRF/lawService.do?OC=kimnahee_fpg&target=efla...


,법령일련번호,현행연혁코드,법령명한글,법령약칭명,법령ID,공포일자,공포번호,제개정구분명,소관부처코드,소관부처명,법령구분명,공동부령정보,시행일자,자법타법여부,법령상세링크
0,285955,현행,신용정보의 이용 및 보호에 관한 법률,신용정보법,001540,20260512,21646,일부개정,1160100,금융위원회,법률,,20260813,,/DRF/lawService.do?OC=kimnahee_fpg&target=efla...
1,220673,현행,신용정보의 이용 및 보호에 관한 법률 시행규칙,신용정보법 시행규칙,007643,20200805,01635,일부개정,1160100,금융위원회,총리령,,20200805,,/DRF/lawService.do?OC=kimnahee_fpg&target=efla...
2,288561,현행,신용정보의 이용 및 보호에 관한 법률 시행령,신용정보법 시행령,004105,20260811,36574,일부개정,1160100,금융위원회,대통령령,,20260813,,/DRF/lawService.do?OC=kimnahee_fpg&target=efla...


In [6]:
# 03. 법령 ID 추출

def get_exact_law_id(search_df, law_name):

    exact = search_df[
        search_df["법령명한글"].str.strip() == law_name
    ]

    if exact.empty:
        raise ValueError(f"{law_name}을 찾지 못했습니다.")

    return exact.iloc[0]["법령ID"]


privacy_id = get_exact_law_id(
    privacy_search,
    "개인정보 보호법"
)

credit_id = get_exact_law_id(
    credit_search,
    "신용정보의 이용 및 보호에 관한 법률"
)

print("개인정보 보호법 ID:", privacy_id)
print("신용정보법 ID:", credit_id)

개인정보 보호법 ID: 011357
신용정보법 ID: 001540


In [7]:
# 04. 법령 전체 조문 수집

def fetch_law_articles(law_id, law_name):

    params = {
        "OC": OC,
        "target": "eflaw",
        "type": "XML",
        "ID": law_id
    }

    response = requests.get(
        SERVICE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    root = ET.fromstring(response.content)

    rows = []

    for article in root.findall(".//조문단위"):

        texts = []

        for node in article.iter():

            if node.tag in [
                "조문내용",
                "항내용",
                "호내용",
                "목내용"
            ]:

                text = "".join(node.itertext()).strip()

                if text:
                    texts.append(text)

        rows.append({
            "law_name": law_name,
            "law_id": law_id,
            "article_no": article.findtext("조문번호"),
            "article_sub_no": article.findtext("조문가지번호"),
            "article_title": article.findtext("조문제목"),
            "effective_date": article.findtext("조문시행일자"),
            "article_text": "\n".join(texts)
        })

    return pd.DataFrame(rows)


In [8]:
privacy_articles = fetch_law_articles(
    privacy_id,
    "개인정보 보호법"
)

credit_articles = fetch_law_articles(
    credit_id,
    "신용정보의 이용 및 보호에 관한 법률"
)

display(privacy_articles.head())
display(credit_articles.head())

,law_name,law_id,article_no,article_sub_no,article_title,effective_date,article_text
0,개인정보 보호법,011357,1,None,,20251002,제1장 총칙
1,개인정보 보호법,011357,1,None,목적,20251002,제1조(목적) 이 법은 개인정보의 처리 및 보호에 관한 사항을 정함으로써 개인의 자...
2,개인정보 보호법,011357,2,None,정의,20251002,제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2014.3.2...
3,개인정보 보호법,011357,3,None,개인정보 보호 원칙,20251002,제3조(개인정보 보호 원칙)\n① 개인정보처리자는 개인정보의 처리 목적을 명확하게 ...
4,개인정보 보호법,011357,4,None,정보주체의 권리,20251002,제4조(정보주체의 권리) 정보주체는 자신의 개인정보 처리와 관련하여 다음 각 호의 ...


,law_name,law_id,article_no,article_sub_no,article_title,effective_date,article_text
0,신용정보의 이용 및 보호에 관한 법률,001540,1,None,,20260813,제1장 총칙
1,신용정보의 이용 및 보호에 관한 법률,001540,1,None,목적,20260813,제1조(목적) 이 법은 신용정보 관련 산업을 건전하게 육성하고 신용정보의 효율적 이...
2,신용정보의 이용 및 보호에 관한 법률,001540,2,None,정의,20260813,제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2011.5.1...
3,신용정보의 이용 및 보호에 관한 법률,001540,3,None,신용정보 관련 산업의 육성,20260813,제3조(신용정보 관련 산업의 육성)\n① 금융위원회는 신용정보 제공능력의 향상과 신...
4,신용정보의 이용 및 보호에 관한 법률,001540,3,2,다른 법률과의 관계,20260813,제3조의2(다른 법률과의 관계)\n① 신용정보의 이용 및 보호에 관하여 다른 법률에...


In [9]:
# 05. Legal Constraint 분석용 원천 데이터 생성

legal_raw = pd.concat(
    [
        privacy_articles,
        credit_articles
    ],
    ignore_index=True
)

print("총 조문 수:", len(legal_raw))

display(legal_raw.head())

총 조문 수: 248


,law_name,law_id,article_no,article_sub_no,article_title,effective_date,article_text
0,개인정보 보호법,011357,1,None,,20251002,제1장 총칙
1,개인정보 보호법,011357,1,None,목적,20251002,제1조(목적) 이 법은 개인정보의 처리 및 보호에 관한 사항을 정함으로써 개인의 자...
2,개인정보 보호법,011357,2,None,정의,20251002,제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2014.3.2...
3,개인정보 보호법,011357,3,None,개인정보 보호 원칙,20251002,제3조(개인정보 보호 원칙)\n① 개인정보처리자는 개인정보의 처리 목적을 명확하게 ...
4,개인정보 보호법,011357,4,None,정보주체의 권리,20251002,제4조(정보주체의 권리) 정보주체는 자신의 개인정보 처리와 관련하여 다음 각 호의 ...


# CHAPTER 3-1. Legal Constraint 후보 조문 추출

전체 법령 조문 중 Financial Privacy Gateway의 데이터 처리 정책과
직접 연결되는 조문을 우선 선별한다.

주요 Constraint 영역은 다음과 같다.

- 목적 제한
- 최소 처리
- 동의
- 제3자 제공
- 처리위탁
- 가명처리
- 재식별 방지
- 안전성 확보조치
- 국외 이전
- 개인신용정보 이용·제공

본 단계에서는 해당 개념과 관련된 조문을 키워드 기반으로 1차 후보군으로 추출한다.

In [10]:
# 06. 분석 불필요 행 제거

legal_clean = legal_raw.copy()

legal_clean = legal_clean[
    legal_clean["article_text"].notna()
].copy()

legal_clean = legal_clean[
    legal_clean["article_title"].notna()
].copy()

print("정제 후 조문 수:", len(legal_clean))

정제 후 조문 수: 248


In [11]:
# 07. Legal Constraint 키워드 정의

constraint_keywords = {
    "PURPOSE_LIMITATION": [
        "처리 목적", "이용 목적", "목적 외"
    ],
    "DATA_MINIMIZATION": [
        "최소한", "최소한의 개인정보", "필요한 범위"
    ],
    "CONSENT": [
        "동의", "정보주체의 동의"
    ],
    "THIRD_PARTY": [
        "제3자", "제공받는 자", "제공하여서는"
    ],
    "OUTSOURCING": [
        "위탁", "수탁자"
    ],
    "PSEUDONYMIZATION": [
        "가명정보", "가명처리", "추가정보"
    ],
    "REIDENTIFICATION": [
        "특정 개인을 알아보기", "재식별"
    ],
    "SECURITY": [
        "안전성 확보", "안전조치", "보호조치"
    ],
    "CROSS_BORDER": [
        "국외", "국외이전"
    ],
    "CREDIT_INFORMATION": [
        "개인신용정보", "신용정보주체"
    ]
}

In [12]:
# 08. 조문별 Constraint 자동 태깅

def detect_constraints(text):

    detected = []

    for constraint, keywords in constraint_keywords.items():

        if any(keyword in text for keyword in keywords):
            detected.append(constraint)

    return detected


legal_clean["constraint_type"] = (
    legal_clean["article_text"]
    .apply(detect_constraints)
)

constraint_df = legal_clean[
    legal_clean["constraint_type"].str.len() > 0
].copy()

print("Constraint 후보 조문:", len(constraint_df))

display(
    constraint_df[
        [
            "law_name",
            "article_no",
            "article_title",
            "constraint_type",
            "article_text"
        ]
    ].head(20)
)

Constraint 후보 조문: 120


,law_name,article_no,article_title,constraint_type,article_text
2,개인정보 보호법,2,정의,[PSEUDONYMIZATION],제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2014.3.2...
3,개인정보 보호법,3,개인정보 보호 원칙,"[PURPOSE_LIMITATION, DATA_MINIMIZATION]",제3조(개인정보 보호 원칙)\n① 개인정보처리자는 개인정보의 처리 목적을 명확하게 ...
4,개인정보 보호법,4,정보주체의 권리,[CONSENT],제4조(정보주체의 권리) 정보주체는 자신의 개인정보 처리와 관련하여 다음 각 호의 ...
5,개인정보 보호법,5,국가 등의 책무,"[PURPOSE_LIMITATION, CONSENT]","제5조(국가 등의 책무)\n① 국가와 지방자치단체는 개인정보의 목적 외 수집, 오용..."
16,개인정보 보호법,7,보호위원회의 심의ㆍ의결 사항 등,[CROSS_BORDER],제7조의9(보호위원회의 심의ㆍ의결 사항 등)\n① 보호위원회는 다음 각 호의 사항을...
18,개인정보 보호법,7,위원의 제척ㆍ기피ㆍ회피,[CONSENT],제7조의11(위원의 제척ㆍ기피ㆍ회피)\n① 위원은 다음 각 호의 어느 하나에 해당하...
31,개인정보 보호법,14,국제협력,[CROSS_BORDER],제14조(국제협력)\n① 정부는 국제적 환경에서의 개인정보 보호 수준을 향상시키기 ...
34,개인정보 보호법,15,개인정보의 수집ㆍ이용,"[PURPOSE_LIMITATION, CONSENT, THIRD_PARTY, SEC...",제15조(개인정보의 수집ㆍ이용)\n① 개인정보처리자는 다음 각 호의 어느 하나에 해...
35,개인정보 보호법,16,개인정보의 수집 제한,"[DATA_MINIMIZATION, CONSENT]",제16조(개인정보의 수집 제한)\n① 개인정보처리자는 제15조제1항 각 호의 어느 ...
36,개인정보 보호법,17,개인정보의 제공,"[PURPOSE_LIMITATION, CONSENT, THIRD_PARTY, SEC...",제17조(개인정보의 제공)\n① 개인정보처리자는 다음 각 호의 어느 하나에 해당되는...


In [13]:
# 09. FPG 직접 관련 핵심 조문 필터링

core_articles = {
    "개인정보 보호법": [
        "3", "15", "16", "17", "18", "19",
        "21", "22", "23", "24", "24의2",
        "26", "28의2", "28의4", "28의5",
        "28의8", "29"
    ],
    "신용정보의 이용 및 보호에 관한 법률": [
        "15", "17", "20", "32", "33",
        "34", "38의2", "40", "40의2"
    ]
}

def normalize_article(row):
    no = str(row["article_no"])

    sub = row["article_sub_no"]

    if pd.notna(sub) and str(sub).strip() not in ["", "None", "0"]:
        return f"{no}의{str(sub)}"

    return no


constraint_df["article_key"] = constraint_df.apply(
    normalize_article,
    axis=1
)

core_constraint_df = constraint_df[
    constraint_df.apply(
        lambda x: x["article_key"]
        in core_articles.get(x["law_name"], []),
        axis=1
    )
].copy()

print("전체 조문:", len(legal_raw))
print("키워드 후보:", len(constraint_df))
print("핵심 Legal Constraint:", len(core_constraint_df))

display(
    core_constraint_df[
        [
            "law_name",
            "article_key",
            "article_title",
            "constraint_type",
            "article_text"
        ]
    ]
)

전체 조문: 248
키워드 후보: 120
핵심 Legal Constraint: 27


,law_name,article_key,article_title,constraint_type,article_text
3,개인정보 보호법,3,개인정보 보호 원칙,"[PURPOSE_LIMITATION, DATA_MINIMIZATION]",제3조(개인정보 보호 원칙)\n① 개인정보처리자는 개인정보의 처리 목적을 명확하게 ...
34,개인정보 보호법,15,개인정보의 수집ㆍ이용,"[PURPOSE_LIMITATION, CONSENT, THIRD_PARTY, SEC...",제15조(개인정보의 수집ㆍ이용)\n① 개인정보처리자는 다음 각 호의 어느 하나에 해...
35,개인정보 보호법,16,개인정보의 수집 제한,"[DATA_MINIMIZATION, CONSENT]",제16조(개인정보의 수집 제한)\n① 개인정보처리자는 제15조제1항 각 호의 어느 ...
36,개인정보 보호법,17,개인정보의 제공,"[PURPOSE_LIMITATION, CONSENT, THIRD_PARTY, SEC...",제17조(개인정보의 제공)\n① 개인정보처리자는 다음 각 호의 어느 하나에 해당되는...
37,개인정보 보호법,18,개인정보의 목적 외 이용ㆍ제공 제한,"[PURPOSE_LIMITATION, CONSENT, THIRD_PARTY, SEC...",제18조(개인정보의 목적 외 이용ㆍ제공 제한)\n① 개인정보처리자는 개인정보를 제1...
38,개인정보 보호법,19,개인정보를 제공받은 자의 이용ㆍ제공 제한,"[PURPOSE_LIMITATION, CONSENT, THIRD_PARTY]",제19조(개인정보를 제공받은 자의 이용ㆍ제공 제한) 개인정보처리자로부터 개인정보를 ...
41,개인정보 보호법,21,개인정보의 파기,"[PURPOSE_LIMITATION, PSEUDONYMIZATION]","제21조(개인정보의 파기)\n① 개인정보처리자는 보유기간의 경과, 개인정보의 처리 ..."
42,개인정보 보호법,22,동의를 받는 방법,"[PURPOSE_LIMITATION, CONSENT]",제22조(동의를 받는 방법)\n① 개인정보처리자는 이 법에 따른 개인정보의 처리에 ...
45,개인정보 보호법,23,민감정보의 처리 제한,"[CONSENT, SECURITY]","제23조(민감정보의 처리 제한)\n①개인정보처리자는 사상ㆍ신념, 노동조합ㆍ정당의 가..."
46,개인정보 보호법,24,고유식별정보의 처리 제한,"[CONSENT, SECURITY]",제24조(고유식별정보의 처리 제한)\n① 개인정보처리자는 다음 각 호의 경우를 제외...


In [14]:
# 10. 장·절 제목 등 비조문 행 제거

core_constraint_clean = core_constraint_df[
    core_constraint_df["article_title"].notna()
].copy()

core_constraint_clean = core_constraint_clean[
    core_constraint_clean["article_title"].str.strip() != ""
].copy()

print("정제 전:", len(core_constraint_df))
print("실제 핵심 조문:", len(core_constraint_clean))

정제 전: 27
실제 핵심 조문: 25


In [15]:
# 11. Constraint 유형별 빈도

constraint_exploded = (
    core_constraint_clean
    .explode("constraint_type")
)

constraint_summary = (
    constraint_exploded
    .groupby("constraint_type")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="article_count")
)

display(constraint_summary)

,constraint_type,article_count
0,CONSENT,14
1,THIRD_PARTY,13
2,PURPOSE_LIMITATION,11
3,SECURITY,9
4,CREDIT_INFORMATION,8
5,PSEUDONYMIZATION,6
6,OUTSOURCING,5
7,DATA_MINIMIZATION,3
8,REIDENTIFICATION,2
9,CROSS_BORDER,1


In [16]:
# 12. 법률 × Constraint 교차표

constraint_crosstab = pd.crosstab(
    constraint_exploded["constraint_type"],
    constraint_exploded["law_name"]
)

display(constraint_crosstab)

law_name,개인정보 보호법,신용정보의 이용 및 보호에 관한 법률
constraint_type,,
CONSENT,11,3
CREDIT_INFORMATION,0,8
CROSS_BORDER,1,0
DATA_MINIMIZATION,2,1
OUTSOURCING,2,3
PSEUDONYMIZATION,4,2
PURPOSE_LIMITATION,9,2
REIDENTIFICATION,2,0
SECURITY,9,0


# CHAPTER 3-3. Legal Constraint 구조 분석

핵심 조문에서는 하나의 법적 의무만 독립적으로 나타나지 않았다.

특히 CONSENT, THIRD_PARTY, PURPOSE_LIMITATION이 반복적으로 나타났으며,
개인정보 보호법에서는 SECURITY,
신용정보법에서는 CREDIT_INFORMATION 및 OUTSOURCING 조건도 함께 나타났다.

이는 금융 데이터의 처리 가능 여부가
단순한 개인정보 여부만으로 결정되지 않고,

Purpose
+ Data Scope
+ Consent
+ Destination
+ Outsourcing
+ Security

등 복수 조건의 조합으로 결정됨을 보여준다.

따라서 Financial Privacy Gateway는
단일 Transform 규칙이 아니라
복수 Legal Constraint를 동시에 평가하는 Policy Engine 구조가 필요하다.

In [17]:
# 13. Constraint 동시출현 분석

from itertools import combinations
from collections import Counter

pair_counter = Counter()

for constraints in core_constraint_clean["constraint_type"]:

    constraints = sorted(set(constraints))

    for pair in combinations(constraints, 2):
        pair_counter[pair] += 1

pair_df = pd.DataFrame(
    [
        {
            "constraint_1": pair[0],
            "constraint_2": pair[1],
            "co_occurrence": count
        }
        for pair, count in pair_counter.items()
    ]
).sort_values(
    "co_occurrence",
    ascending=False
)

display(pair_df.head(15))

,constraint_1,constraint_2,co_occurrence
3,CONSENT,THIRD_PARTY,9
1,CONSENT,PURPOSE_LIMITATION,8
5,PURPOSE_LIMITATION,THIRD_PARTY,8
2,CONSENT,SECURITY,7
4,PURPOSE_LIMITATION,SECURITY,5
6,SECURITY,THIRD_PARTY,5
24,CREDIT_INFORMATION,THIRD_PARTY,5
12,OUTSOURCING,THIRD_PARTY,4
16,PSEUDONYMIZATION,THIRD_PARTY,4
9,CONSENT,OUTSOURCING,3


# CHAPTER 3-4. Legal Constraint 동시출현 분석

핵심 Legal Constraint 조문을 대상으로 동시출현 관계를 분석한 결과,

- CONSENT × THIRD_PARTY: 9건
- CONSENT × PURPOSE_LIMITATION: 8건
- PURPOSE_LIMITATION × THIRD_PARTY: 8건
- CONSENT × SECURITY: 7건

순으로 높은 동시출현이 나타났다.

이는 금융 데이터 처리에 대한 법적 판단이 단일 조건으로 결정되지 않고,
여러 조건이 동시에 충족되어야 하는 복합 Constraint 구조임을 보여준다.

특히 외부 AI·SaaS로 데이터를 전달하는 상황에서는

Purpose
→ Consent
→ Third-party / Outsourcing
→ Security

조건을 독립적으로 판단하는 것이 아니라
하나의 요청 단위에서 동시에 검증할 필요가 있다.

따라서 FPG의 정책 판단 구조 역시
단일 데이터 유형 기반 규칙이 아니라
복수 Constraint를 결합하는 Policy Engine으로 설계할 필요가 있다.

In [19]:
# 14. Constraint 연관성 강도 계산

from itertools import combinations
import pandas as pd

transactions = core_constraint_clean["constraint_type"].apply(set).tolist()
n_docs = len(transactions)

all_constraints = sorted(
    set().union(*transactions)
)

support_count = {
    c: sum(c in t for t in transactions)
    for c in all_constraints
}

rows = []

for c1, c2 in combinations(all_constraints, 2):

    both = sum(
        (c1 in t) and (c2 in t)
        for t in transactions
    )

    if both == 0:
        continue

    union = sum(
        (c1 in t) or (c2 in t)
        for t in transactions
    )

    support_xy = both / n_docs
    support_x = support_count[c1] / n_docs
    support_y = support_count[c2] / n_docs

    jaccard = both / union

    lift = support_xy / (
        support_x * support_y
    )

    rows.append({
        "constraint_1": c1,
        "constraint_2": c2,
        "co_occurrence": both,
        "jaccard": round(jaccard, 3),
        "lift": round(lift, 3)
    })

association_df = pd.DataFrame(rows)

association_df = association_df.sort_values(
    ["lift", "jaccard", "co_occurrence"],
    ascending=False
)

display(association_df.head(20))


,constraint_1,constraint_2,co_occurrence,jaccard,lift
14,CROSS_BORDER,OUTSOURCING,1,0.200,5.000
23,PSEUDONYMIZATION,REIDENTIFICATION,2,0.333,4.167
15,CROSS_BORDER,SECURITY,1,0.111,2.778
10,CREDIT_INFORMATION,OUTSOURCING,3,0.300,1.875
1,CONSENT,CROSS_BORDER,1,0.071,1.786
18,OUTSOURCING,PSEUDONYMIZATION,2,0.222,1.667
21,OUTSOURCING,THIRD_PARTY,4,0.286,1.538
27,PURPOSE_LIMITATION,THIRD_PARTY,8,0.500,1.399
7,CONSENT,SECURITY,7,0.438,1.389
5,CONSENT,PURPOSE_LIMITATION,8,0.471,1.299


In [20]:
# 15. 의미 있는 Constraint 조합 추출

strong_pairs = association_df[
    (association_df["lift"] > 1) &
    (association_df["co_occurrence"] >= 2)
].copy()

display(strong_pairs)

,constraint_1,constraint_2,co_occurrence,jaccard,lift
23,PSEUDONYMIZATION,REIDENTIFICATION,2,0.333,4.167
10,CREDIT_INFORMATION,OUTSOURCING,3,0.300,1.875
18,OUTSOURCING,PSEUDONYMIZATION,2,0.222,1.667
21,OUTSOURCING,THIRD_PARTY,4,0.286,1.538
27,PURPOSE_LIMITATION,THIRD_PARTY,8,0.500,1.399
7,CONSENT,SECURITY,7,0.438,1.389
5,CONSENT,PURPOSE_LIMITATION,8,0.471,1.299
25,PSEUDONYMIZATION,THIRD_PARTY,4,0.267,1.282
26,PURPOSE_LIMITATION,SECURITY,5,0.333,1.263
8,CONSENT,THIRD_PARTY,9,0.500,1.236


# CHAPTER 3-6. Legal Constraint 결합구조 해석

Lift 분석 결과 가장 높은 값은
PSEUDONYMIZATION × REIDENTIFICATION (Lift=4.167)이었다.

다만 해당 조합의 동시출현은 2건으로,
희소한 Constraint의 특성상 Lift가 크게 나타난 결과일 수 있다.

따라서 FPG 정책 설계에서는 Lift만 단독으로 사용하지 않고
동시출현 빈도와 함께 해석하였다.

그 결과 주요 결합 구조는 다음과 같다.

- PURPOSE_LIMITATION × THIRD_PARTY: 8건, Lift 1.399
- CONSENT × SECURITY: 7건, Lift 1.389
- CONSENT × PURPOSE_LIMITATION: 8건, Lift 1.299
- CONSENT × THIRD_PARTY: 9건, Lift 1.236
- CREDIT_INFORMATION × OUTSOURCING: 3건, Lift 1.875

즉 외부 AI·SaaS 데이터 처리 판단은
단순히 개인정보 여부나 동의 여부 하나만 확인하는 구조가 아니다.

목적, 제공 대상, 동의, 정보유형, 위탁 여부, 보안조치를
동시에 평가해야 하는 복합 Constraint 구조가 확인되었다.

따라서 FPG의 Policy Engine은
단일 조건 기반 Rule이 아니라
복수 조건의 결합에 따라 처리방식을 결정하는 구조로 설계한다.

In [21]:
# 16. Legal Constraint → Gateway 판단 구조 정의

constraint_policy_map = {
    "PURPOSE_LIMITATION": "CHECK_PURPOSE",
    "DATA_MINIMIZATION": "MINIMIZE_FIELDS",
    "CONSENT": "CHECK_CONSENT",
    "THIRD_PARTY": "CHECK_DESTINATION",
    "OUTSOURCING": "CHECK_PROCESSOR",
    "PSEUDONYMIZATION": "TRANSFORM",
    "REIDENTIFICATION": "BLOCK_REIDENTIFICATION",
    "SECURITY": "CHECK_SECURITY",
    "CROSS_BORDER": "CHECK_REGION",
    "CREDIT_INFORMATION": "CHECK_CREDIT_SCOPE"
}

constraint_exploded["gateway_control"] = (
    constraint_exploded["constraint_type"]
    .map(constraint_policy_map)
)

display(
    constraint_exploded[
        [
            "law_name",
            "article_key",
            "article_title",
            "constraint_type",
            "gateway_control"
        ]
    ]
)

,law_name,article_key,article_title,constraint_type,gateway_control
3,개인정보 보호법,3,개인정보 보호 원칙,PURPOSE_LIMITATION,CHECK_PURPOSE
3,개인정보 보호법,3,개인정보 보호 원칙,DATA_MINIMIZATION,MINIMIZE_FIELDS
34,개인정보 보호법,15,개인정보의 수집ㆍ이용,PURPOSE_LIMITATION,CHECK_PURPOSE
34,개인정보 보호법,15,개인정보의 수집ㆍ이용,CONSENT,CHECK_CONSENT
34,개인정보 보호법,15,개인정보의 수집ㆍ이용,THIRD_PARTY,CHECK_DESTINATION
...,...,...,...,...,...
219,신용정보의 이용 및 보호에 관한 법률,40,신용정보회사등의 금지사항,CREDIT_INFORMATION,CHECK_CREDIT_SCOPE
220,신용정보의 이용 및 보호에 관한 법률,40의2,가명처리ㆍ익명처리에 관한 행위규칙,THIRD_PARTY,CHECK_DESTINATION
220,신용정보의 이용 및 보호에 관한 법률,40의2,가명처리ㆍ익명처리에 관한 행위규칙,OUTSOURCING,CHECK_PROCESSOR
220,신용정보의 이용 및 보호에 관한 법률,40의2,가명처리ㆍ익명처리에 관한 행위규칙,PSEUDONYMIZATION,TRANSFORM


In [22]:
# 17. 조문별 Gateway Control 결합

article_controls = (
    constraint_exploded
    .groupby([
        "law_name",
        "article_key",
        "article_title"
    ])["gateway_control"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="gateway_controls")
)

article_controls["control_count"] = (
    article_controls["gateway_controls"].str.len()
)

display(
    article_controls.sort_values(
        "control_count",
        ascending=False
    )
)

,law_name,article_key,article_title,gateway_controls,control_count
20,신용정보의 이용 및 보호에 관한 법률,32,개인신용정보의 제공ㆍ활용에 대한 동의,"[CHECK_CONSENT, CHECK_CREDIT_SCOPE, CHECK_DEST...",6
10,개인정보 보호법,26,업무위탁에 따른 개인정보의 처리 제한,"[CHECK_CONSENT, CHECK_DESTINATION, CHECK_PROCE...",5
3,개인정보 보호법,18,개인정보의 목적 외 이용ㆍ제공 제한,"[CHECK_CONSENT, CHECK_DESTINATION, CHECK_PURPO...",4
2,개인정보 보호법,17,개인정보의 제공,"[CHECK_CONSENT, CHECK_DESTINATION, CHECK_PURPO...",4
21,신용정보의 이용 및 보호에 관한 법률,33,개인신용정보의 이용,"[CHECK_CONSENT, CHECK_CREDIT_SCOPE, CHECK_DEST...",4
11,개인정보 보호법,28의2,가명정보의 처리 등,"[BLOCK_REIDENTIFICATION, CHECK_CONSENT, CHECK_...",4
12,개인정보 보호법,28의4,가명정보에 대한 안전조치의무 등,"[CHECK_DESTINATION, CHECK_PURPOSE, CHECK_SECUR...",4
24,신용정보의 이용 및 보호에 관한 법률,40의2,가명처리ㆍ익명처리에 관한 행위규칙,"[CHECK_CREDIT_SCOPE, CHECK_DESTINATION, CHECK_...",4
0,개인정보 보호법,15,개인정보의 수집ㆍ이용,"[CHECK_CONSENT, CHECK_DESTINATION, CHECK_PURPO...",4
17,신용정보의 이용 및 보호에 관한 법률,15,수집 및 처리의 원칙,"[CHECK_CONSENT, CHECK_CREDIT_SCOPE, CHECK_DEST...",4


In [23]:
# 18. Gateway Control 조합 분석

control_pattern = (
    article_controls
    .assign(
        control_pattern=
        article_controls["gateway_controls"]
        .apply(lambda x: " + ".join(x))
    )
    .groupby("control_pattern")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

display(control_pattern.head(15))

,control_pattern,article_count
12,CHECK_CREDIT_SCOPE,3
7,CHECK_CONSENT + CHECK_DESTINATION + CHECK_PURP...,3
10,CHECK_CONSENT + CHECK_SECURITY,2
0,BLOCK_REIDENTIFICATION + CHECK_CONSENT + CHECK...,1
3,CHECK_CONSENT + CHECK_CREDIT_SCOPE + CHECK_DES...,1
2,CHECK_CONSENT + CHECK_CREDIT_SCOPE + CHECK_DES...,1
1,BLOCK_REIDENTIFICATION + TRANSFORM,1
5,CHECK_CONSENT + CHECK_DESTINATION + CHECK_PROC...,1
6,CHECK_CONSENT + CHECK_DESTINATION + CHECK_PURPOSE,1
8,CHECK_CONSENT + CHECK_PROCESSOR + CHECK_REGION...,1


# CHAPTER 3-7. Legal Constraint → Gateway Control 구조화

법령 조문을 Gateway Control로 변환한 결과,
하나의 조문에서 하나의 Control만 요구되는 것이 아니라
복수 Control이 동시에 요구되는 구조가 확인된다.

예를 들어 외부 데이터 처리 요청은

CHECK_PURPOSE
+ CHECK_CONSENT
+ CHECK_DESTINATION
+ CHECK_SECURITY

와 같이 복수 조건을 동시에 평가할 수 있다.

따라서 FPG의 정책 판단 단위는 개별 Control이 아니라
요청별 Control Set으로 설계한다.

단, 법령은 MASK, HMAC, TOKEN 등의 구체적 기술을 직접 지정하지 않는다.

법령 분석은 "어떤 의무를 만족해야 하는가"를 결정하고,
실제 Transform 방법은 Privacy–Utility 실험 결과와
기관 내부 정책을 결합하여 선택한다.

즉,

Legal Constraint
→ Required Controls
→ Policy Condition
→ Transform Candidate
→ Privacy–Utility 기준 선택

의 구조로 연결한다.